# Ingeniería de datos

Para la ingeniería de datos se usará ´scikit-learn´, particularmente las herramientas de Pipeline y ColumnTransformer.

## Pipeline

Un Pipeline en scikit-learn es como una línea de ensamblaje en una fábrica: una secuencia de pasos donde cada paso transforma los datos y los pasa al siguiente.

🧠 **¿Para qué sirve?**

* Para encadenar varios pasos de preprocesamiento y modelado (por ejemplo: imputación de valores faltantes → escalado → modelo).

* Para automatizar y evitar errores en el flujo de trabajo.

* Para facilitar la integración con técnicas de validación cruzada y grid search.

## ColumnTransformer

Un ColumnTransformer te permite aplicar transformaciones distintas a columnas diferentes. Por ejemplo, escalar solo columnas numéricas y aplicar codificación one-hot a las categóricas.

🧠 **¿Para qué sirve?**

* Para hacer preprocesamiento por tipo de variable sin separar el dataset.

* Para integrar el preprocesamiento con un modelo dentro de un pipeline.

# Pasos para un Feature Engineering

In [27]:
# base libraries for data science
from pathlib import Path

import pandas as pd
import sklearn as sk
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [28]:
DATA_DIR = Path.cwd().resolve().parent / "datos"

datos_titanic = pd.read_parquet(DATA_DIR / "02_datos_con_tipo_de_dato_ajustado_titanic.parquet", engine="pyarrow")

In [29]:
print("Pandas version: ", pd.__version__)
print("sklearn version: ", sk.__version__)

Pandas version:  2.2.3
sklearn version:  1.6.1


In [30]:
datos_titanic.columns

Index(['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked'],
      dtype='object')

In [31]:
datos_titanic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   pclass    1309 non-null   int64   
 1   survived  1309 non-null   bool    
 2   name      1309 non-null   object  
 3   sex       1309 non-null   category
 4   age       1046 non-null   float64 
 5   sibsp     1309 non-null   int8    
 6   parch     1309 non-null   int8    
 7   fare      1308 non-null   float64 
 8   embarked  1307 non-null   category
dtypes: bool(1), category(2), float64(2), int64(1), int8(2), object(1)
memory usage: 47.7+ KB


## Eliminación de Columna Name

In [9]:
# Al ser una columna poco relevante, la eliminamos

columnas_seleccionadas = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked",
    "survived",
]

In [10]:
titanic_features = datos_titanic[columnas_seleccionadas].copy()
titanic_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   pclass    1309 non-null   int64   
 1   sex       1309 non-null   category
 2   age       1046 non-null   float64 
 3   sibsp     1309 non-null   int8    
 4   parch     1309 non-null   int8    
 5   fare      1308 non-null   float64 
 6   embarked  1307 non-null   category
 7   survived  1309 non-null   bool    
dtypes: bool(1), category(2), float64(2), int64(1), int8(2)
memory usage: 37.5 KB


## Valores Faltantes

In [11]:
titanic_features.isna().sum()

pclass        0
sex           0
age         263
sibsp         0
parch         0
fare          1
embarked      2
survived      0
dtype: int64

## Datos Duplicados

In [12]:
filas_duplicadas = titanic_features.duplicated().sum()

In [14]:
print("Cantidad de filas duplicadas: ", filas_duplicadas)

Cantidad de filas duplicadas:  195


In [15]:
titanic_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   pclass    1309 non-null   int64   
 1   sex       1309 non-null   category
 2   age       1046 non-null   float64 
 3   sibsp     1309 non-null   int8    
 4   parch     1309 non-null   int8    
 5   fare      1308 non-null   float64 
 6   embarked  1307 non-null   category
 7   survived  1309 non-null   bool    
dtypes: bool(1), category(2), float64(2), int64(1), int8(2)
memory usage: 37.5 KB


## Ingeniería de Datos

In [16]:
# Codificar la variable objetivo

titanic_features["survived"] = titanic_features["survived"].astype("int")

titanic_features.sample(2)

,pclass,sex,age,sibsp,parch,fare,embarked,survived
258,1,female,30.0,0,0,31.00,C,1
1179,3,male,NaN,1,9,69.55,S,0


In [22]:
# Categoricas ordinales y target

target = 'survived'

cols_categoric_ord = ['pclass'] 

In [25]:
# Seleccionar columnas categóricas

cols_numeric = titanic_features.select_dtypes(include=['int64', 'float64', 'int8']).columns.tolist()
cols_numeric = [col for col in cols_numeric if col not in [target] + cols_categoric_ord]
cols_numeric

['age', 'sibsp', 'parch', 'fare']

In [26]:
# Columnas categóricas (no ordenadas)

cols_categoric = titanic_features.select_dtypes(include=['object', 'category']).columns.tolist()
cols_categoric

['sex', 'embarked']

## Pipelines

In [33]:
numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

In [35]:
categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder()),
    ]
)

In [36]:
categorical_ord_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OrdinalEncoder()),
    ]
)

## Preprocesador

In [37]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, cols_numeric),
        ("categoric", categorical_pipe, cols_categoric),
        ("categoric ordinales", categorical_ord_pipe, cols_categoric_ord),
    ]
)

preprocessor

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['age', 'sibsp', 'parch', 'fare']),
                                ('categoric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot', OneHotEncoder())]),
                                 ['sex', 'embarked']),
                                ('categoric ordinales',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot', OrdinalEncoder())]),
                                 ['pclass'])])

## Ejemplo de entrenamiento de aplicación del preprocesador

### Train / Test split

In [38]:
# conjunto de entrenamiento y de prueba

X_features = titanic_features.drop("survived", axis="columns")
Y_target = titanic_features["survived"]

In [40]:
# 80% train, 20% test

x_train, x_test, y_train, y_test = train_test_split(
    X_features, Y_target, test_size=0.2, stratify=Y_target
)

In [43]:
# Verificamos dimensiones

x_train.shape, y_train.shape

((1047, 7), (1047,))

In [ ]:
# Verificamos dimensiones

x_test.shape, y_test.shape

((262, 7), (262,))

#### Preprocesamiento

In [44]:
transformed_data = preprocessor.fit(x_train)

transformed_data

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['age', 'sibsp', 'parch', 'fare']),
                                ('categoric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot', OneHotEncoder())]),
                                 ['sex', 'embarked']),
                                ('categoric ordinales',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot', OrdinalEncoder())]),
                                 ['pclass'])])

In [46]:
feature_names = preprocessor.get_feature_names_out()

feature_names

array(['numeric__age', 'numeric__sibsp', 'numeric__parch',
       'numeric__fare', 'categoric__sex_female', 'categoric__sex_male',
       'categoric__embarked_C', 'categoric__embarked_Q',
       'categoric__embarked_S', 'categoric ordinales__pclass'],
      dtype=object)

Para ver con más detalle las transformaciones del one hote encoding, podemos convertir rápidamente en un dataframe las variables preprocesadas:

In [48]:
# Convertir a DataFrame para mejor visualización


x_train_transformed = preprocessor.transform(x_train) # Lo aplico a los datos de entrenamiento

x_train_transformed = pd.DataFrame(x_train_transformed, columns=feature_names) # Lo convierto en un DataFrame para visualizar

x_train_transformed.info() # Chequeamos rápidamente

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1047 entries, 0 to 1046
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   numeric__age                 1047 non-null   float64
 1   numeric__sibsp               1047 non-null   float64
 2   numeric__parch               1047 non-null   float64
 3   numeric__fare                1047 non-null   float64
 4   categoric__sex_female        1047 non-null   float64
 5   categoric__sex_male          1047 non-null   float64
 6   categoric__embarked_C        1047 non-null   float64
 7   categoric__embarked_Q        1047 non-null   float64
 8   categoric__embarked_S        1047 non-null   float64
 9   categoric ordinales__pclass  1047 non-null   float64
dtypes: float64(10)
memory usage: 81.9 KB


In [49]:
x_train_transformed.head(2)

,numeric__age,numeric__sibsp,numeric__parch,numeric__fare,categoric__sex_female,categoric__sex_male,categoric__embarked_C,categoric__embarked_Q,categoric__embarked_S,categoric ordinales__pclass
0,1.0,5.0,2.0,46.9,0.0,1.0,0.0,0.0,1.0,2.0
1,23.0,1.0,0.0,10.5,0.0,1.0,0.0,0.0,1.0,1.0


In [50]:
x_train.head(2)

,pclass,sex,age,sibsp,parch,fare,embarked
826,3,male,1.0,5,2,46.9,S
579,2,male,23.0,1,0,10.5,S


# Recomendaciones e Ideas  

## Manejo de Datos Faltantes  
**📌 Recomendación:**  
Evalúa el impacto de diferentes estrategias de imputación en el rendimiento del modelo. Considera técnicas avanzadas como `KNNImputer` o `IterativeImputer`.  

**🔍 Fundamento:**  
Distintas estrategias de imputación pueden afectar de manera diferente el desempeño del modelo. Técnicas avanzadas podrían proporcionar estimaciones más precisas para los valores faltantes.  

---  
## Escalado de Variables Numéricas  
**📌 Recomendación:**  
Añade un paso de escalado en el pipeline numérico usando `StandardScaler` o `MinMaxScaler`.  

**🔍 Fundamento:**  
Escalar las variables numéricas mejora el rendimiento de muchos algoritmos de machine learning, asegurando que todas contribuyan de manera equilibrada al modelo.  

---  
## Ingeniería de Características  
**📌 Recomendación:**  
Explora técnicas para crear nuevas variables a partir de las existentes. Por ejemplo:  
- Combinar `sibsp` y `parch` para crear `family_size`.  

**🔍 Fundamento:**  
La ingeniería de características captura patrones no evidentes en los datos originales, mejorando la capacidad predictiva del modelo.  

---  
**💡 Nota:**  
Los términos técnicos (`KNNImputer`, `StandardScaler`, etc.) se conservan en inglés por convención en machine learning.  